# 동의 기반 웹캠 얼굴 식별

얼굴 식별(Face Identification)은 카메라에 보이는 얼굴을 **미리 등록한 사람들의 얼굴 특징**과 비교해 가장 유사한 등록자를 찾는 작업이다. YOLO가 사람의 위치를 찾더라도 그 사람이 누구인지는 알 수 없으므로 얼굴 검출과 얼굴 특징 비교가 별도로 필요하다.

이 실습에서는 YOLO가 `person`을 탐지하고 임시 track ID를 유지한다. 같은 frame에서 YuNet이 얼굴 위치와 5개 landmark를 찾고, SFace가 정렬된 얼굴을 `(1, 128)` embedding으로 바꾼다. 등록 embedding과의 cosine similarity가 기준을 통과한 결과가 여러 frame에서 반복될 때만 이름을 표시하고, 나머지는 `Unknown`으로 둔다.

이 노트북은 동의한 사람을 로컬에서 잠시 구분하는 수업용 폐쇄형 데모이다. 얼굴 원본과 embedding을 파일로 저장하지 않으며 출석, 출입 통제, 본인 인증이나 임의 인물 검색에는 사용하지 않는다.

## 사람 탐지에서 등록자 식별까지의 흐름

사람을 찾는 YOLO, 얼굴을 찾는 YuNet, 얼굴을 비교하는 SFace는 서로 다른 출력을 만든다. 세 결과를 같은 frame 좌표와 최근 track 기록으로 연결해야 최종 이름을 표시할 수 있다.

```mermaid
flowchart LR
    A[웹캠 frame] --> B[YOLO person 탐지와 추적]
    B --> C[person box와 임시 track ID]
    A --> D[YuNet 얼굴 검출]
    D --> E[얼굴 box와 5개 landmark]
    E --> F[alignCrop 얼굴 정렬]
    F --> G[SFace embedding]
    G --> H[등록 embedding과 cosine 비교]
    C --> I[얼굴 중심이 포함된 person box 연결]
    H --> I
    I --> J[최근 5 frame 판정]
    J --> K[이름 또는 Unknown 표시]
```

- **사람 탐지**: YOLO가 사람 전체의 위치를 `person` bounding box로 찾는 단계이다.
- **얼굴 검출**: YuNet이 얼굴 box와 두 눈·코·입꼬리의 5개 landmark를 찾는 단계이다.
- **얼굴 정렬**: 기울어진 얼굴의 landmark를 기준 위치에 맞춰 SFace가 비교하기 쉬운 형태로 바꾸는 단계이다.
- **얼굴 embedding**: 얼굴 모양의 특징을 현재 SFace 모델 기준 `(1, 128)` 숫자 벡터로 표현한 결과이다.
- **Cosine similarity**: 두 embedding의 방향이 얼마나 비슷한지 나타내며 값이 클수록 같은 얼굴일 가능성이 높다.
- **Track ID**: 연속 frame에서 같은 사람 box를 연결하기 위한 임시 번호이며 실제 신원이나 이름을 의미하지 않는다.
- **Unknown**: 등록자와 충분히 비슷하지 않거나 얼굴이 보이지 않아 신원을 확정하지 않은 안전한 기본값이다.

## 실행 환경과 개인정보 경계

웹캠은 로컬 PC에 연결되어 있으므로 PyCharm 또는 로컬 Jupyter 커널에서 실행한다. RunPod와 Colab의 `VideoCapture(0)`은 학생 PC가 아닌 원격 서버의 카메라를 찾으므로 이 코드로는 로컬 웹캠을 사용할 수 없다. 등록 창에서는 얼굴이 한 명만 보이게 한 뒤 `Space`를 눌러 샘플을 수집하고, 실시간 식별 창에서는 `q`를 눌러 종료한다.

얼굴 embedding도 원본 사진을 대신해 사람을 구분할 수 있는 생체정보이다. 이 실습은 다음 원칙을 지킨다.

- 촬영 대상에게 목적과 실행 시간을 알리고 명시적인 동의를 받는다.
- 공개해도 되는 영문 별칭만 사용하며 얼굴 frame과 embedding을 출력·저장·전송하지 않는다.
- 등록되지 않은 얼굴을 자동으로 추가하지 않고 항상 `Unknown`으로 처리한다.
- 감정, 성별, 나이, 성격과 같은 민감속성을 추론하지 않는다.
- 사진이나 휴대전화 화면을 실제 사람으로 오인할 수 있으므로 인증·출석·출입 판단에 사용하지 않는다.

실습에서는 동의한 1~2명이 한 명씩 화면에 등장하도록 진행한다. 여러 사람이 겹치거나 얼굴이 가려지면 track ID가 바뀌고 오인식 위험이 커질 수 있다.

## 실습 패키지 설치

Ultralytics는 YOLO 사람 탐지와 ByteTrack 추적을 제공하고, OpenCV는 웹캠·YuNet·SFace·화면 표시를 담당한다. 앞 노트북과 같은 버전을 사용해 두 실습의 실행 환경을 맞춘다. 설치 후 import 오류가 계속되면 커널을 한 번 재시작한다.

In [ ]:
%pip install -q ultralytics==8.4.135 opencv-python==4.10.0.84

## 라이브러리와 판정 기준 설정

`PERSON_CONFIDENCE`는 YOLO 사람 box를 남기는 기준이고, `FACE_SCORE_THRESHOLD`는 YuNet 얼굴 검출 기준이다. 두 값은 서로 다른 모델의 점수이므로 혼용하지 않는다.

OpenCV 공식 SFace 예제는 LFW 데이터셋에서 cosine 기준 `0.363`을 사용하지만 데이터셋마다 적절한 값이 다르다. 이 실습은 오탐을 줄이기 위한 보수적인 시작값 `0.45`를 사용하고, 등록자가 둘 이상이면 1등과 2등 점수 차이도 확인한다. 최종 이름은 최근 5개 frame 중 같은 이름이 3회 이상 나온 경우에만 표시한다.

In [ ]:
from collections import Counter, deque
from pathlib import Path
from urllib.request import urlretrieve

import cv2
import numpy as np
from ultralytics import YOLO

# 카메라와 YOLO 설정은 앞 노트북과 같은 CPU 실습 조건을 사용한다.
CAMERA_INDEX = 0
YOLO_MODEL_ID = "yolo26n.pt"
PERSON_CLASS_ID = 0
PERSON_CONFIDENCE = 0.40
YOLO_IMAGE_SIZE = 480
DEVICE = "cpu"

# YuNet 검출 기준과 SFace 식별 안정화 기준은 서로 다른 단계에 적용된다.
FACE_SCORE_THRESHOLD = 0.90
FACE_NMS_THRESHOLD = 0.30
FACE_TOP_K = 100
COSINE_THRESHOLD = 0.45
MARGIN_THRESHOLD = 0.05
HISTORY_SIZE = 5
REQUIRED_MATCHES = 3

# cv2.putText는 한글 표시가 제한되므로 공개 가능한 영문 별칭을 사용한다.
REGISTER_NAME = "teacher"
REGISTRATION_SAMPLE_COUNT = 5

print({
    "camera_index": CAMERA_INDEX,
    "registered_alias": REGISTER_NAME,
    "cosine_threshold": COSINE_THRESHOLD,
    "history_rule": f"{REQUIRED_MATCHES}/{HISTORY_SIZE}",
})

## YuNet과 SFace 모델 다운로드

YuNet은 얼굴 box와 5개 landmark를 반환하고, SFace는 정렬된 얼굴을 embedding으로 변환한다. OpenCV 4.x와 맞는 모델 조합을 사용하기 위해 OpenCV Zoo `4.10.0` 태그의 `face_detection_yunet_2023mar.onnx`와 `face_recognition_sface_2021dec.onnx`를 `models` 폴더에 내려받는다. SFace 파일은 약 37MB이므로 최초 실행에는 네트워크 시간이 필요하다. 다운로드가 중단된 파일을 모델로 잘못 사용하지 않도록 태그에 고정된 파일 크기도 확인한다.

YuNet 모델 디렉터리는 MIT, SFace 모델 디렉터리는 Apache-2.0 조건을 사용한다. 수업 밖에서 배포하거나 서비스에 적용할 때는 [YuNet](https://github.com/opencv/opencv_zoo/tree/4.10.0/models/face_detection_yunet)과 [SFace](https://github.com/opencv/opencv_zoo/tree/4.10.0/models/face_recognition_sface)의 모델별 라이선스를 다시 확인한다.

In [ ]:
# 모델 파일은 개인 절대경로가 아닌 현재 커널 기준의 상대 models 폴더에 둔다.
MODEL_DIR = Path("models")
YUNET_MODEL_PATH = MODEL_DIR / "face_detection_yunet_2023mar.onnx"
SFACE_MODEL_PATH = MODEL_DIR / "face_recognition_sface_2021dec.onnx"

# main branch가 바뀌어도 수업 파일이 달라지지 않도록 OpenCV Zoo 4.10.0 태그 URL을 사용한다.
YUNET_MODEL_URL = (
    "https://github.com/opencv/opencv_zoo/raw/4.10.0/models/"
    "face_detection_yunet/face_detection_yunet_2023mar.onnx"
)
SFACE_MODEL_URL = (
    "https://github.com/opencv/opencv_zoo/raw/4.10.0/models/"
    "face_recognition_sface/face_recognition_sface_2021dec.onnx"
)
YUNET_MODEL_SIZE = 232_589
SFACE_MODEL_SIZE = 38_696_353

# 크기가 맞는 모델은 재사용하고, 새 파일은 .part에 완전히 받은 뒤 실제 경로로 옮긴다.
def download_verified(url, destination, expected_size):
    if destination.exists() and destination.stat().st_size == expected_size:
        return destination
    temporary_path = destination.with_suffix(destination.suffix + ".part")
    urlretrieve(url, temporary_path)
    if temporary_path.stat().st_size != expected_size:
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError(f"모델 다운로드 크기가 올바르지 않다: {destination.name}")
    temporary_path.replace(destination)
    return destination

MODEL_DIR.mkdir(parents=True, exist_ok=True)
# 반환된 두 경로는 바로 다음 셀의 FaceDetectorYN과 FaceRecognizerSF 입력으로 사용된다.
model_paths = [
    download_verified(YUNET_MODEL_URL, YUNET_MODEL_PATH, YUNET_MODEL_SIZE),
    download_verified(SFACE_MODEL_URL, SFACE_MODEL_PATH, SFACE_MODEL_SIZE),
]
print({path.name: round(path.stat().st_size / 1024**2, 2) for path in model_paths})

## YOLO·YuNet·SFace 초기화

`YOLO()`는 사람 탐지와 추적에 사용할 가중치를 읽는다. `FaceDetectorYN.create()`는 YuNet ONNX 파일과 얼굴 검출 기준을 받아 detector를 만들고, `FaceRecognizerSF.create()`는 SFace ONNX 파일을 읽어 얼굴 정렬·embedding·비교 기능을 준비한다.

`registered_face_db`에는 `영문 별칭 → 여러 embedding`만 메모리에서 연결한다. 이 셀을 다시 실행하면 이전 등록 내용이 사라지므로 모델 준비 후 한 번만 실행한다.

In [ ]:
person_tracker = YOLO(YOLO_MODEL_ID)

# 초기 input size는 placeholder이며 실제 frame마다 setInputSize로 현재 너비와 높이를 전달한다.
face_detector = cv2.FaceDetectorYN.create(
    str(YUNET_MODEL_PATH),
    "",
    (320, 320),
    FACE_SCORE_THRESHOLD,
    FACE_NMS_THRESHOLD,
    FACE_TOP_K,
)
face_recognizer = cv2.FaceRecognizerSF.create(str(SFACE_MODEL_PATH), "")

# 얼굴 원본과 embedding을 파일로 저장하지 않고 현재 커널의 dictionary에서만 유지한다.
registered_face_db = {}
match_history_by_track = {}

print({
    "yolo_model": YOLO_MODEL_ID,
    "yunet_model": YUNET_MODEL_PATH.name,
    "sface_model": SFACE_MODEL_PATH.name,
    "registered_names": list(registered_face_db),
})

## 웹캠 frame에서 얼굴 embedding 추출

YuNet의 `detect()`는 얼굴이 있으면 `(N, 15)` 배열을 반환한다. `N`은 얼굴 수이고, 각 행에는 얼굴의 `x, y, width, height`, 5개 landmark의 좌표 10개와 검출 score가 들어 있다. 얼굴이 없으면 `None`이 반환된다.

SFace의 `alignCrop()`은 YuNet landmark를 사용해 얼굴을 정렬하고, `feature()`는 현재 모델에서 `(1, 128)` embedding을 만든다. 단순히 얼굴 사각형만 잘라 크기를 바꾸면 눈·코 위치가 맞지 않아 비교 품질이 낮아질 수 있으므로 정렬 단계를 생략하지 않는다.

In [ ]:
def read_required_frame(camera):
    success, frame = camera.read()
    if not success:
        raise RuntimeError("웹캠 frame을 읽지 못했다. 카메라 권한과 CAMERA_INDEX를 확인한다.")
    return frame

# YuNet input size의 순서는 NumPy shape의 (H, W)와 달리 (W, H)이다.
def detect_faces(frame_bgr):
    frame_height, frame_width = frame_bgr.shape[:2]
    face_detector.setInputSize((frame_width, frame_height))
    _, faces = face_detector.detect(frame_bgr)
    return faces

# face_row의 box와 5개 landmark로 얼굴을 정렬한 뒤 (1, 128) 특징을 복사해 반환한다.
def extract_face_feature(frame_bgr, face_row):
    aligned_face = face_recognizer.alignCrop(frame_bgr, face_row)
    return face_recognizer.feature(aligned_face).copy()

## 동의한 등록자의 얼굴 샘플 수집 함수

등록 창에는 한 명의 얼굴만 보이게 한다. 얼굴 box가 나타나면 정면, 약간 왼쪽, 약간 오른쪽처럼 자세를 조금 바꾸면서 `Space`를 눌러 5개 embedding을 수집한다. 원본 frame은 저장하지 않으며 `q`를 누르면 등록을 취소하고 카메라를 해제한다.

여러 샘플을 사용하는 이유는 한 장의 조명·표정·각도에만 맞춘 비교를 줄이기 위해서이다. 다만 샘플 수가 많아도 인증 정확도를 보장할 수는 없다.

In [ ]:
def capture_registration_features(display_name, sample_count):
    # samples에는 얼굴 원본이 아니라 SFace가 만든 embedding 배열만 임시로 쌓인다.
    samples = []
    camera = cv2.VideoCapture(CAMERA_INDEX)

    # 등록 완료·취소·오류 중 어느 경로로 끝나도 finally에서 카메라를 반환한다.
    try:
        while len(samples) < sample_count:
            frame_bgr = read_required_frame(camera)
            faces = detect_faces(frame_bgr)
            preview = frame_bgr.copy()
            # 정확히 한 얼굴일 때만 누가 등록되는지 모호하지 않은 유효 샘플로 본다.
            valid_face = faces is not None and len(faces) == 1

            if valid_face:
                face_row = faces[0]
                x, y, width, height = map(int, face_row[:4])
                cv2.rectangle(preview, (x, y), (x + width, y + height), (0, 255, 0), 2)
                status = f"SPACE capture {len(samples)}/{sample_count}"
            else:
                status = "Show exactly one face"

            cv2.putText(preview, f"Register {display_name} | {status} | Q cancel",
                        (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
            cv2.imshow("Face Registration", preview)
            # Space는 현재 정렬 얼굴을 등록하고 q는 샘플을 남기지 않은 채 취소한다.
            pressed_key = cv2.waitKey(1) & 0xFF

            if pressed_key == ord("q"):
                raise RuntimeError("사용자가 얼굴 등록을 취소했다.")
            if pressed_key == ord(" ") and valid_face:
                samples.append(extract_face_feature(frame_bgr, face_row))
    finally:
        camera.release()
        cv2.destroyAllWindows()

    return samples

## 등록자 embedding을 메모리에 추가

이 셀을 실행하면 `Face Registration` 창이 열린다. 동의한 등록자 한 명만 카메라 앞에 서고 얼굴 box가 초록색으로 보일 때 `Space`를 5번 누른다. 같은 별칭으로 다시 실행하면 이전 샘플을 덮어쓰며, 다른 사람을 추가하려면 위 설정 셀의 `REGISTER_NAME`을 다른 영문 별칭으로 바꾼 뒤 이 셀만 다시 실행한다.

출력에는 embedding 숫자를 표시하지 않고 등록된 별칭과 샘플 개수만 표시한다.

In [ ]:
# display_name은 화면 별칭이고 sample_count는 서로 다른 자세에서 모을 embedding 수이다.
registered_face_db[REGISTER_NAME] = capture_registration_features(
    display_name=REGISTER_NAME,
    sample_count=REGISTRATION_SAMPLE_COUNT,
)

# embedding 값은 출력하지 않고 등록 상태만 확인한다.
print({
    "registered_names": list(registered_face_db),
    "sample_counts": {name: len(samples) for name, samples in registered_face_db.items()},
})

## 등록 embedding과 실시간 얼굴 비교

실시간 얼굴 embedding은 등록자별 5개 embedding과 각각 cosine similarity를 계산하고 평균 점수로 순위를 만든다. 가장 높은 점수가 `COSINE_THRESHOLD` 이상이어야 하며, 등록자가 둘 이상이면 1등과 2등의 차이도 `MARGIN_THRESHOLD` 이상이어야 이름 후보가 된다. 기준을 하나라도 통과하지 못하면 `Unknown`이다.

한 frame의 후보만으로 이름을 바로 표시하지 않는다. 현재 frame도 같은 등록자이고 track ID별 최근 5개 후보에서 그 이름이 3회 이상 나온 경우에만 안정된 이름으로 반환한다. 현재 얼굴이 없거나 비교를 통과하지 못하면 즉시 `Unknown`으로 돌아간다. 이는 순간적인 오인식을 줄이는 규칙이며 실제 신원을 보장하는 인증 절차로 사용할 수 없다.

In [ ]:
def identify_face(query_feature):
    # ranking은 등록자별 평균 cosine 점수와 별칭을 묶어 높은 점수부터 비교하기 위한 목록이다.
    ranking = []
    for name, registered_samples in registered_face_db.items():
        # 한 등록자의 여러 자세 샘플과 query를 모두 비교해 한 장의 우연한 점수 영향을 줄인다.
        sample_scores = [
            face_recognizer.match(sample, query_feature, cv2.FaceRecognizerSF_FR_COSINE)
            for sample in registered_samples
        ]
        ranking.append((float(np.mean(sample_scores)), name))

    ranking.sort(reverse=True)
    if not ranking:
        return "Unknown", 0.0, 0.0

    # 최고 점수는 절대 기준을, 1등과 2등 차이는 등록자 사이의 모호함을 검사한다.
    best_score, best_name = ranking[0]
    second_score = ranking[1][0] if len(ranking) > 1 else None
    margin = best_score - second_score if second_score is not None else float("inf")
    passed_margin = second_score is None or margin >= MARGIN_THRESHOLD
    accepted = best_score >= COSINE_THRESHOLD and passed_margin
    return (best_name if accepted else "Unknown"), best_score, margin

# 현재 frame의 후보가 등록자일 때만 최근 기록에서 같은 별칭의 반복 횟수를 센다.
def stabilized_label(recent_labels):
    if not recent_labels:
        return "Unknown"
    current_label = recent_labels[-1]
    if current_label == "Unknown":
        return "Unknown"
    current_count = Counter(recent_labels)[current_label]
    return current_label if current_count >= REQUIRED_MATCHES else "Unknown"

## YOLO 사람 추적과 얼굴 연결

`model.track(..., persist=True)`는 연속된 frame임을 tracker에 알려 이전 사람 box의 임시 ID를 이어 간다. 이 ID는 화면 밖으로 나갔다가 다시 들어오거나 다른 사람과 겹치면 바뀔 수 있으므로 이름 대신 사용할 수 없다.

YOLO person box와 YuNet face box는 같은 원본 frame 좌표를 사용한다. 얼굴 box는 사람 box보다 훨씬 작아 IoU가 낮으므로, 얼굴 중심점이 포함된 person box를 찾는다. 여러 person box가 같은 중심점을 포함하면 면적이 가장 작은 box를 선택해 겹친 사람 사이의 연결을 줄인다.

`track()`의 `source`는 현재 BGR frame, `persist`는 이전 frame의 추적 상태 유지 여부, `tracker`는 ByteTrack 설정이다. `classes`는 사람 class만 선택하고, `conf`는 최소 탐지 신뢰도, `imgsz`는 추론 입력 크기, `device`는 실행 장치, `verbose`는 frame별 로그 출력 여부를 지정한다.

In [ ]:
def track_people(frame_bgr):
    # 현재 frame에서 사람 class만 찾고, ByteTrack의 이전 상태를 이어 임시 ID를 얻는다.
    result = person_tracker.track(
        source=frame_bgr,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[PERSON_CLASS_ID],
        conf=PERSON_CONFIDENCE,
        imgsz=YOLO_IMAGE_SIZE,
        device=DEVICE,
        verbose=False,
    )[0]

    # 사람이 없거나 tracker가 ID를 만들지 못한 frame은 빈 box와 ID 목록으로 반환한다.
    if result.boxes.id is None:
        return np.empty((0, 4), dtype=np.float32), []
    # xyxy box와 track ID를 CPU의 일반 Python·NumPy 값으로 바꿔 다음 함수에 전달한다.
    person_boxes = result.boxes.xyxy.cpu().numpy()
    track_ids = result.boxes.id.int().cpu().tolist()
    return person_boxes, track_ids

def associate_face_with_person(face_row, person_boxes):
    # YuNet의 얼굴 box 중심점이 어느 YOLO 사람 box 안에 들어가는지 확인한다.
    face_x, face_y, face_width, face_height = face_row[:4]
    center_x = face_x + face_width / 2
    center_y = face_y + face_height / 2
    candidates = [
        index for index, (x1, y1, x2, y2) in enumerate(person_boxes)
        if x1 <= center_x <= x2 and y1 <= center_y <= y2
    ]
    if not candidates:
        return None
    # box가 겹치면 더 작은 사람 box를 골라 주변 사람에게 얼굴이 잘못 연결될 가능성을 줄인다.
    return min(candidates, key=lambda index: np.prod(person_boxes[index, 2:] - person_boxes[index, :2]))

## frame별 얼굴 비교 결과 만들기

한 frame에서 각 사람 track의 기본 후보는 `Unknown`이다. 얼굴이 검출되고 person box와 연결된 경우에만 SFace embedding을 비교하며, 같은 person box에 여러 얼굴이 연결되면 가장 높은 similarity 결과 하나만 사용한다.

얼굴 box는 원본 화면에 표시할 수 있도록 frame 경계 안으로 제한한다. 이 단계의 출력은 `track ID별 이름 후보`, `track ID별 최고 similarity`, `표시할 얼굴 box`이며 다음 시각화 함수가 사용한다.

In [ ]:
def match_faces_to_tracks(frame_bgr, faces, person_boxes, track_ids):
    # 얼굴 비교에 성공하기 전에는 모든 사람의 기본 이름을 Unknown으로 둔다.
    candidate_labels = {track_id: "Unknown" for track_id in track_ids}
    candidate_scores = {track_id: 0.0 for track_id in track_ids}
    face_rectangles = []
    frame_height, frame_width = frame_bgr.shape[:2]

    # 이 frame에 얼굴이 없으면 사람 추적 결과는 유지하되 이름 판정은 수행하지 않는다.
    if faces is None:
        return candidate_labels, candidate_scores, face_rectangles

    for face_row in faces:
        person_index = associate_face_with_person(face_row, person_boxes)
        if person_index is None:
            continue

        # 얼굴을 정렬해 SFace embedding으로 바꾸고 등록 샘플과 cosine similarity를 비교한다.
        query_feature = extract_face_feature(frame_bgr, face_row)
        label, score, _ = identify_face(query_feature)
        track_id = track_ids[person_index]
        # 한 사람 box에 얼굴 후보가 여러 개 연결되면 similarity가 가장 높은 결과만 남긴다.
        if score > candidate_scores[track_id]:
            candidate_labels[track_id] = label
            candidate_scores[track_id] = score

        # YuNet의 (x, y, width, height)를 화면 표시용 (x1, y1, x2, y2)로 변환한다.
        x, y, width, height = map(int, face_row[:4])
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(frame_width - 1, x + width), min(frame_height - 1, y + height)
        face_rectangles.append((x1, y1, x2, y2))

    return candidate_labels, candidate_scores, face_rectangles

## 최근 판정 안정화와 화면 표시

각 활성 track에는 최대 5개의 최근 이름 후보만 보관한다. 화면에서 사라진 track 기록은 즉시 제거하고, 현재 frame의 이름 후보가 최근 5개 중 3회 이상 반복되어야 초록색 이름을 표시한다. 현재 얼굴이 없거나 이름이 안정되지 않으면 주황색 `Unknown`을 표시한다.

track ID에 이름을 한 번 저장한 뒤 계속 재사용하지 않는 이유는 ID switch가 발생하면 다른 사람에게 이전 이름이 전달될 수 있기 때문이다. 이 함수는 매 frame 새 얼굴 비교 결과로 짧은 기록을 갱신한다.

In [ ]:
def draw_identification_frame(frame_bgr, person_boxes, track_ids, candidate_labels,
                              candidate_scores, face_rectangles, histories):
    output_frame = frame_bgr.copy()
    active_track_ids = set(track_ids)

    # 화면에서 사라진 track의 이름 기록을 제거해 새 사람에게 이전 이름이 남지 않게 한다.
    for stale_track_id in set(histories) - active_track_ids:
        del histories[stale_track_id]

    # 각 활성 track에는 현재 frame의 후보를 정확히 한 번만 추가한다.
    for track_id in track_ids:
        history = histories.setdefault(track_id, deque(maxlen=HISTORY_SIZE))
        history.append(candidate_labels.get(track_id, "Unknown"))

    for x1, y1, x2, y2 in face_rectangles:
        cv2.rectangle(output_frame, (x1, y1), (x2, y2), (255, 255, 0), 2)

    for box, track_id in zip(person_boxes, track_ids):
        x1, y1, x2, y2 = map(int, box)
        stable_name = stabilized_label(histories[track_id])
        similarity = candidate_scores.get(track_id, 0.0)
        color = (0, 200, 0) if stable_name != "Unknown" else (0, 165, 255)
        label_text = f"{stable_name} | ID {track_id} | sim {similarity:.2f}"
        cv2.rectangle(output_frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(output_frame, label_text, (x1, max(25, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.60, color, 2)

    return output_frame

## 웹캠에서 실시간 등록자 식별

이 셀은 새로운 웹캠 stream에 이전 tracker 상태가 섞이지 않도록 YOLO 객체를 다시 만든다. 이후 `frame 읽기 → YOLO person 추적 → YuNet 얼굴 검출 → SFace 비교 → 최근 판정 안정화 → 화면 표시`를 반복한다. 반복 코드를 함수 안에서 실행해 마지막 얼굴 frame이 Jupyter 전역 변수로 남지 않게 한다.

사람 box에는 `이름 또는 Unknown`, 임시 `ID`, 현재 frame의 `sim`이 표시된다. 얼굴이 옆이나 뒤를 향하거나 가려지면 YOLO는 사람을 계속 찾더라도 이름은 `Unknown`으로 돌아갈 수 있다. 별도 창을 클릭한 뒤 영문 `q` 또는 `Q`를 누르면 카메라와 창이 해제된다.

In [ ]:
# 등록 샘플이 없으면 모든 결과가 Unknown이므로 실행 순서 오류를 먼저 알린다.
if not registered_face_db:
    raise RuntimeError("먼저 동의한 등록자의 얼굴 embedding을 등록한다.")

def run_live_identification(histories):
    camera = cv2.VideoCapture(CAMERA_INDEX)
    pressed_key = -1
    frame_bgr = faces = display_frame = None

    try:
        while pressed_key not in (ord("q"), ord("Q")):
            # 한 frame마다 사람 추적과 얼굴 검출을 독립적으로 수행한다.
            frame_bgr = read_required_frame(camera)
            person_boxes, track_ids = track_people(frame_bgr)
            faces = detect_faces(frame_bgr)
            # 얼굴을 사람 track에 연결한 뒤 등록자 이름 후보와 similarity를 얻는다.
            candidate_labels, candidate_scores, face_rectangles = match_faces_to_tracks(
                frame_bgr, faces, person_boxes, track_ids
            )
            # 현재 후보가 최근 5개 중 3회 이상 일치할 때만 등록자 이름을 표시한다.
            display_frame = draw_identification_frame(
                frame_bgr, person_boxes, track_ids, candidate_labels,
                candidate_scores, face_rectangles, histories
            )
            cv2.putText(display_frame, "Consent-based demo | Q/q quit", (20, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
            # OpenCV 창을 갱신하고 영문 q 또는 Q 입력을 1ms 간격으로 확인한다.
            cv2.imshow("YOLO + YuNet + SFace", display_frame)
            pressed_key = cv2.waitKey(1) & 0xFF
    finally:
        # 성공·취소·오류와 관계없이 frame 참조와 카메라·창을 정리한다.
        frame_bgr = faces = display_frame = None
        camera.release()
        cv2.destroyAllWindows()

# 새 webcam stream마다 tracker와 최근 판정 기록을 새 상태로 시작한다.
person_tracker = YOLO(YOLO_MODEL_ID)
match_history_by_track.clear()
try:
    run_live_identification(match_history_by_track)
finally:
    # Ultralytics predictor가 마지막 입력을 참조하지 않도록 stream 종료 후 해제한다.
    person_tracker.predictor = None

## 등록 embedding과 최근 판정 기록 정리

수업이 끝나면 등록 embedding 배열을 0으로 덮어쓰고 dictionary와 track 기록의 참조를 비운다. 모델 객체도 마지막 입력의 내부 버퍼를 참조할 가능성이 있으므로 함께 해제한다. Python과 운영체제의 메모리 복사 때문에 이 코드만으로 완전한 secure deletion을 보장할 수는 없다. 완전한 메모리 초기화가 필요하면 이 셀을 실행한 뒤 Jupyter 커널을 재시작한다.

In [ ]:
# 등록 샘플 배열을 가능한 범위에서 0으로 덮어써 원래 특징값을 남기지 않는다.
for registered_samples in registered_face_db.values():
    for feature in registered_samples:
        feature.fill(0)

# embedding DB와 짧은 track 판정 기록의 Python 참조를 모두 제거한다.
registered_face_db.clear()
match_history_by_track.clear()

# 모델 객체의 참조도 제거하며 다시 실습하려면 모델 초기화 셀부터 실행한다.
person_tracker = None
face_detector = None
face_recognizer = None
print({
    "registered_names": list(registered_face_db),
    "track_history_count": 0,
    "models_released": True,
})

## 정리와 한계

전체 흐름은 `웹캠 → YOLO person·track ID → YuNet face·landmark → SFace embedding → 등록자 cosine 비교 → 최근 frame 안정화 → 이름 또는 Unknown`이다. YOLO의 track ID는 잠시 같은 사람 box를 연결하고, SFace embedding 비교가 등록자 후보를 정한다. 두 역할을 합치지 않는 것이 핵심이다.

공식 예제의 cosine `0.363`은 LFW 데이터셋에서 얻은 기준이며 카메라, 조명, 자세와 등록자 구성에 따라 오탐과 미탐이 달라진다. 수업에서는 등록자와 비등록자가 각각 등장해 `sim` 분포를 관찰한 뒤 threshold를 조정해야 한다. threshold를 낮추면 등록자를 더 잘 찾을 수 있지만 비등록자를 잘못 이름 붙일 위험도 커진다.

이 구현에는 liveness detection이 없어 사진·영상 재생 공격을 막지 못하고, 가림·재진입·사람 교차 상황에서 track ID가 바뀔 수 있다. 따라서 수업용 시각적 데모로만 사용하며 인증, 출석, 채용, 평가, 감시와 같은 중요한 판단에는 사용하지 않는다.